# Storey-BH versus BH

`select()` applies Benjamini-Hochberg (BH) by default. BH controls the FDR at `pi0 * alpha`, where `pi0` is the share of true nulls, so it is conservative when anomalies are a sizable share of the batch. `method="storey"` estimates `pi0` from the p-values and runs BH at the level `alpha / pi0_hat`.

This notebook runs both rules on one labeled batch and shows where the difference comes from.

In [1]:
import numpy as np
from sklearn.ensemble import IsolationForest

from nonconform import ConformalDetector, Split
from nonconform.metrics import false_discovery_rate, statistical_power

## A batch where anomalies are common

30% of the 200 test points are anomalies, so `pi0 = 0.7`. The labels `y_test` are used only to evaluate the decisions.

In [2]:
rng = np.random.default_rng(42)
x_train = rng.normal(size=(3_000, 4))
x_test = np.vstack([rng.normal(size=(140, 4)), rng.normal(loc=1.75, size=(60, 4))])
y_test = np.r_[np.zeros(140, dtype=int), np.ones(60, dtype=int)]
alpha = 0.1

detector = ConformalDetector(
    detector=IsolationForest(random_state=42),
    strategy=Split(n_calib=1_000),
    seed=42,
).fit(x_train)

## Compare the two rules

Both calls score the same batch against the same 1,000 calibration scores. Choose the rule before looking at the batch; running both and reporting the better one is not covered by either guarantee.

In [3]:
bh_selected = np.asarray(detector.select(x_test, alpha=alpha))
storey_selected = np.asarray(detector.select(x_test, alpha=alpha, method="storey"))

for name, selected in [("BH", bh_selected), ("Storey-BH", storey_selected)]:
    print(
        f"{name}: {int(selected.sum())} discoveries, "
        f"realized FDP {false_discovery_rate(y_test, selected):.3f}, "
        f"power {statistical_power(y_test, selected):.3f}"
    )

BH: 40 discoveries, realized FDP 0.050, power 0.633
Storey-BH: 43 discoveries, realized FDP 0.047, power 0.683


## Where the difference comes from

Storey-BH adjusted p-values are the BH adjusted p-values scaled by `pi0_hat` and capped at one. Thresholding either at `alpha` reproduces the masks above, and their ratio recovers the estimate. The estimate is deliberately conservative (it errs toward one), so Storey-BH runs only somewhat above `alpha` here and the gain is modest. It grows with the share of anomalies.

In [4]:
result = detector.last_result
bh_adjusted = result.adjusted_p_values()
storey_adjusted = result.adjusted_p_values(method="storey")

np.testing.assert_array_equal(bh_adjusted <= alpha, bh_selected)
np.testing.assert_array_equal(storey_adjusted <= alpha, storey_selected)

smallest = np.argmin(bh_adjusted)
pi0_hat = storey_adjusted[smallest] / bh_adjusted[smallest]
print(f"estimated pi0: {pi0_hat:.3f} (true: {1 - y_test.mean():.2f})")
print(f"Storey-BH runs BH at alpha / pi0_hat = {alpha / pi0_hat:.3f}")

estimated pi0: 0.841 (true: 0.70)
Storey-BH runs BH at alpha / pi0_hat = 0.119


Storey-BH controls the FDR at `alpha` for classical `Empirical` p-values from `Split` or detached calibration with exchangeable, continuous scores. It also accepts `ConditionalEmpirical` with classical ties, and it rejects randomized, KDE-based, weighted, and `DerandomizedSplits` evidence. See the FDR control guide for the full scope.